# Modelo de produção — risco de alfabetização municipal

Este notebook usa o Random Forest ajustado sem log, selecionado no notebook 02. Ele reproduz o painel Gold e a separação original, mede o desempenho no teste de 2024 e treina o pipeline que será usado para pontuar novos dados.

A unidade de análise é município e rede. O alvo vale 1 quando a taxa oficial fica abaixo da meta do ano. Os atributos vêm do ano anterior. A amostra de desenvolvimento contempla a rede 3.

In [2]:
from pathlib import Path
from importlib.metadata import version

import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, average_precision_score, confusion_matrix,
    f1_score, precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEMENTE = 42
LIMIAR = 0.50
raiz = Path.cwd()
if raiz.name == 'notebooks':
    raiz = raiz.parent

municipios = pd.read_csv(raiz / 'data' / 'gold' / 'indicadores_municipio.csv')

## Painel histórico e pipeline

A junção usa a chave município–rede e exige um registro por chave em cada ano. O modelo recebe apenas os indicadores de 2023 e o código de rede. A taxa e a meta de 2024 definem o rótulo.

In [4]:
# ============================================================
# 1. COLUNAS HISTÓRICAS - 2023
# ============================================================

colunas_historicas = [
    'id_municipio',
    'ano',
    'taxa_oficial',
    'alunos_presentes',
    'percentual_participacao',
    'nivel_alfabetizacao',
    'media_proficiencia',
    'media_portugues',
    'rede'
]

# Filtrar histórico de 2023
historico = municipios.loc[
    municipios['ano'].eq(2023),
    colunas_historicas
].copy()

# Renomear variáveis históricas
historico = historico.rename(
    columns={
        coluna: f'{coluna}_2023'
        for coluna in colunas_historicas
        if coluna not in ['id_municipio', 'rede']
    }
)

# ============================================================
# 2. DADOS DO ANO ALVO - 2024
# ============================================================

resultado = municipios.loc[
    municipios['ano'].eq(2024),
    [
        'id_municipio',
        'rede',
        'ano',
        'taxa_oficial',
        'meta_do_ano'
    ]
].copy()

resultado = resultado.rename(
    columns={
        'ano': 'ano_alvo',
        'taxa_oficial': 'taxa_2024',
        'meta_do_ano': 'meta_2024'
    }
)

# ============================================================
# 3. UNIFICAR HISTÓRICO E RESULTADO
# ============================================================

painel = historico.merge(
    resultado,
    on=['id_municipio', 'rede'],
    how='inner',
    validate='one_to_one'
)

# ============================================================
# 4. REMOVER REGISTROS SEM VARIÁVEL ALVO
# ============================================================

painel_completo = painel.dropna(
    subset=['taxa_2024', 'meta_2024']
).copy()

# ============================================================
# 5. CRIAR VARIÁVEL ALVO - CLASSIFICAÇÃO
# ============================================================

painel_completo['risco_meta'] = (
    painel_completo['taxa_2024'] < painel_completo['meta_2024']
).astype(int)

# 0 = Município atingiu ou superou a meta
# 1 = Município não atingiu a meta

# ============================================================
# 6. ORDENAR E REORGANIZAR ÍNDICE
# ============================================================

painel_completo = (
    painel_completo
    .sort_values(['id_municipio', 'rede'])
    .reset_index(drop=True)
)

# ============================================================
# 7. VALIDAÇÕES
# ============================================================

assert not painel_completo.duplicated(
    subset=['id_municipio', 'rede']
).any(), 'Existem chaves duplicadas.'

assert painel_completo['ano_2023'].lt(
    painel_completo['ano_alvo']
).all()

assert painel_completo['risco_meta'].isin([0, 1]).all()


In [5]:
# ============================================================
# 1. VARIÁVEIS NUMÉRICAS
# ============================================================

colunas_numericas = [
    'taxa_oficial_2023',
    'alunos_presentes_2023',
    'percentual_participacao_2023',
    'nivel_alfabetizacao_2023',
    'media_proficiencia_2023',
    'media_portugues_2023'
]

# ============================================================
# 2. VARIÁVEIS CATEGÓRICAS
# ============================================================

colunas_categoricas = [
    'rede'
]

# ============================================================
# 3. VARIÁVEL ALVO
# ============================================================

coluna_alvo = 'risco_meta'

# ============================================================
# 4. FEATURES E TARGET
# ============================================================

features = colunas_numericas + colunas_categoricas

# ============================================================
# 3. FEATURES E TARGET
# ============================================================

X = painel_completo[features].copy()
y = painel_completo[coluna_alvo].copy()

In [6]:
def criar_preprocessador():

    numerico = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

    categorico = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore'))
    ])

    return ColumnTransformer([
        ('num', numerico, colunas_numericas),
        ('cat', categorico, colunas_categoricas)
    ])

pipeline_rf = Pipeline([
    ('preprocessor', criar_preprocessador()),
    ('model', RandomForestClassifier(
        n_estimators=80,
        max_depth=8,
        min_samples_leaf=15,
        max_features='sqrt',
        class_weight='balanced_subsample',
        random_state=SEMENTE,
        n_jobs=-1
    ))
])

## Avaliação no teste reservado

Recriamos os mesmos 923 registros de teste do notebook 02. Com a configuração já escolhida, treinamos em treino mais validação e medimos o resultado no teste. O limiar de 0,50 vem da regra de classificação do notebook 02. Esta avaliação separa municípios no mesmo período de referência. A validação em um ano futuro depende de resultados observados desse ano.

In [8]:
indices_desenvolvimento, indices_teste = train_test_split(
    painel_completo.index.to_numpy(), test_size=0.20,
    random_state=SEMENTE, stratify=y
)
assert set(painel_completo.loc[indices_desenvolvimento, 'id_municipio']).isdisjoint(
    painel_completo.loc[indices_teste, 'id_municipio']
)

modelo_avaliacao = clone(pipeline_rf).fit(X.loc[indices_desenvolvimento], y.loc[indices_desenvolvimento])
prob_teste = modelo_avaliacao.predict_proba(X.loc[indices_teste])[:, 1]
pred_teste = (prob_teste >= LIMIAR).astype(int)
y_teste = y.loc[indices_teste]

metricas = pd.Series({
    'Acurácia': accuracy_score(y_teste, pred_teste),
    'Precisão': precision_score(y_teste, pred_teste, zero_division=0),
    'Recall': recall_score(y_teste, pred_teste, zero_division=0),
    'F1': f1_score(y_teste, pred_teste, zero_division=0),
    'ROC-AUC': roc_auc_score(y_teste, prob_teste),
    'Precisão média (AP)': average_precision_score(y_teste, prob_teste),
}, name='Teste reservado')

display(metricas.round(4))
display(pd.DataFrame(
    confusion_matrix(y_teste, pred_teste, labels=[0, 1]),
    index=['Real 0', 'Real 1'], columns=['Previsto 0', 'Previsto 1']
))

Acurácia               0.6143
Precisão               0.5795
Recall                 0.6200
F1                     0.5991
ROC-AUC                0.6618
Precisão média (AP)    0.6369
Name: Teste reservado, dtype: float64

,Previsto 0,Previsto 1
Real 0,301,193
Real 1,163,266


## Treinamento final e exportação

Após a avaliação, o pipeline de produção é reajustado com todos os registros rotulados para aproveitar também os dados do teste. As métricas acima pertencem ao pipeline avaliado antes desse reajuste. O arquivo `modelos/risco_alfabetizacao.joblib` inclui o pré-processamento, o classificador, o esquema de entrada, o limiar e as versões do ambiente. Carregue apenas artefatos de origem confiável.

In [10]:
modelo_producao = clone(pipeline_rf).fit(X, y)
artefato = {
    'pipeline': modelo_producao,
    'features': features,
    'limiar': LIMIAR,
    'ano_historico_treino': 2023,
    'ano_alvo_treino': 2024,
    'redes_treino': sorted(X['rede'].unique().tolist()),
    'versoes': {nome: version(nome) for nome in ['scikit-learn', 'numpy', 'pandas', 'joblib']},
}

caminho_modelo = raiz / 'modelos' / 'risco_alfabetizacao.joblib'
caminho_modelo.parent.mkdir(exist_ok=True)
caminho_temporario = caminho_modelo.with_suffix('.tmp')
joblib.dump(artefato, caminho_temporario, compress=3)
caminho_temporario.replace(caminho_modelo)

carregado = joblib.load(caminho_modelo)
np.testing.assert_allclose(
    carregado['pipeline'].predict_proba(X.head()),
    modelo_producao.predict_proba(X.head())
)
print(caminho_modelo.relative_to(raiz))

modelos\risco_alfabetizacao.joblib


## Inferência em lote — cenário 2025

Pontuamos os municípios da rede coberta usando os indicadores Gold de 2024. O sufixo `_2023` faz parte do esquema aprendido no treino. Na inferência, cada coluna com esse sufixo recebe o indicador correspondente do ano anterior ao ano previsto.

O ranking é uma estimativa para 2025. Sua interpretação depende da disponibilidade dos indicadores no momento da previsão e da estabilidade da relação aprendida entre 2023 e 2024.

In [12]:
def pontuar_municipios(dados_gold, ano_historico, artefato):
    chaves = ['id_municipio', 'rede']
    renomear = {
        atributo.removesuffix('_2023'): atributo
        for atributo in artefato['features'] if atributo.endswith('_2023')
    }
    base = dados_gold.loc[
        dados_gold['ano'].eq(ano_historico)
        & dados_gold['rede'].isin(artefato['redes_treino']),
        chaves + list(renomear)
    ].rename(columns=renomear)
    if base.empty:
        raise ValueError('Nenhum registro elegível para o ano e as redes do modelo.')
    if base[chaves].isna().any().any() or base.duplicated(chaves).any():
        raise ValueError('A chave município–rede deve ser completa e única no lote.')

    pipeline = artefato['pipeline']
    coluna_risco = pipeline.classes_.tolist().index(1)
    probabilidades = pipeline.predict_proba(base[artefato['features']])[:, coluna_risco]
    return base[chaves].assign(
        ano_alvo=ano_historico + 1,
        probabilidade_risco=probabilidades,
        risco_previsto=(probabilidades >= artefato['limiar']).astype(int)
    ).sort_values(
        ['probabilidade_risco', 'id_municipio', 'rede'],
        ascending=[False, True, True]
    ).reset_index(drop=True)

ranking_2025 = pontuar_municipios(municipios, 2024, carregado)
print(f'{len(ranking_2025):,} registros pontuados')
display(ranking_2025.head(10))

5,449 registros pontuados


,id_municipio,rede,ano_alvo,probabilidade_risco,risco_previsto
0,1501402,3,2025,0.853067,1
1,4309209,3,2025,0.848120,1
2,1500602,3,2025,0.845751,1
3,3301702,3,2025,0.845574,1
4,3305109,3,2025,0.844824,1
5,4304606,3,2025,0.844247,1
6,3300456,3,2025,0.843090,1
7,1200203,3,2025,0.842009,1
8,3304904,3,2025,0.840840,1
9,4316907,3,2025,0.836686,1
